In [2]:
import csv
import time
import logging
from pathlib import Path
from urllib.parse import urlparse
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# --- Configuration ---
BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/fazwaz')
START_URL = "https://www.fazwaz.co.th/โครงการทั้งหมด/กรุงเทพมหานคร"
OUTPUT_CSV_FILE = BASE_DIR / 'fazwaz_listing_urls.csv'

MAX_PAGES = 20
PAGE_TIMEOUT = 15

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver() -> uc.Chrome:
    """Configures Chrome for maximum text-scraping performance."""
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager' # โหลดแค่ DOM ไม่รอรูปภาพ
    
    # Block ภาพและ CSS เพื่อลด Bandwidth และ Overhead ของ CPU
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    
    return uc.Chrome(options=options, version_main=145)


def extract_links(driver: uc.Chrome, base_url: str) -> list:
    """Extracts property URLs using fast JavaScript execution."""
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('.site-map-item a.site-map-item-link'))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)
    return [base_url + h if h.startswith("/") else h for h in hrefs]


def scroll_to_bottom(driver: uc.Chrome):
    """Scrolls down to trigger any lazy-loaded content or pagination buttons."""
    last_h = 0
    for _ in range(6): 
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.5)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h


def click_next_page(driver: uc.Chrome, wait: WebDriverWait) -> bool:
    """Finds and clicks the 'Next' pagination button."""
    try:
        next_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//a[@rel='next']")))
        
        # เลื่อนลงไปหาปุ่มก่อนกด เพื่อหลีกเลี่ยง ElementClickInterceptedException
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
        time.sleep(0.5)
        
        driver.execute_script("arguments[0].click();", next_btn)
        return True
    except (TimeoutException, NoSuchElementException):
        return False


def main():
    # สร้าง Directory ถ้ายังไม่มี
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    
    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    base_url = f"{urlparse(START_URL).scheme}://{urlparse(START_URL).netloc}"
    
    all_urls = set()
    current_page = 1

    logging.info(f"Starting scrape at: {START_URL}")
    driver.get(START_URL)

    try:
        while current_page <= MAX_PAGES:
            logging.info(f"Processing Page {current_page}/{MAX_PAGES}...")
            
            # รอจนกว่ากล่องข้อมูลจะปรากฏ
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.site-map-item')))
            except TimeoutException:
                logging.warning(f"Timeout waiting for elements on page {current_page}. Stopping.")
                break 

            scroll_to_bottom(driver)
            
            new_links = extract_links(driver, base_url)
            if not new_links:
                logging.info("No links found on this page. Stopping.")
                break
                
            all_urls.update(new_links)
            logging.info(f"Found {len(new_links)} listings (Total unique: {len(all_urls)})")

            if current_page >= MAX_PAGES:
                logging.info(f"Reached target max pages ({MAX_PAGES}).")
                break

            # ---------------------------------------------------------
            # FIX: วิธีแก้ปัญหา TimeoutException แบบฉบับ SPA (Single Page App)
            # ---------------------------------------------------------
            current_url = driver.current_url
            
            if not click_next_page(driver, wait):
                logging.info("No 'Next' button found. Reached the last page.")
                break
                
            # รอจนกว่า URL เก่าจะเปลี่ยนไป (แปลว่า React จัดการ Update State แล้ว)
            try:
                wait.until(EC.url_changes(current_url))
                # รอโหลดการ์ดชุดใหม่ให้เสร็จ
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.site-map-item')))
            except TimeoutException:
                logging.error("Timeout waiting for the next page to load.")
                break
                
            current_page += 1
            
    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    # บันทึกข้อมูลแบบ I/O คลีนๆ
    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in sorted(all_urls):
            writer.writerow([u])
            
    logging.info(f"✓ Successfully saved {len(all_urls)} URLs to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

2026-03-29 14:29:50,529 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 14:29:51,224 - INFO - Starting scrape at: https://www.fazwaz.co.th/โครงการทั้งหมด/กรุงเทพมหานคร
2026-03-29 14:29:53,540 - INFO - Processing Page 1/20...
2026-03-29 14:29:55,602 - INFO - Found 30 listings (Total unique: 30)
2026-03-29 14:29:58,226 - INFO - Processing Page 2/20...
2026-03-29 14:29:59,254 - INFO - Found 30 listings (Total unique: 60)
2026-03-29 14:30:01,838 - INFO - Processing Page 3/20...
2026-03-29 14:30:02,874 - INFO - Found 30 listings (Total unique: 90)
2026-03-29 14:30:05,968 - INFO - Processing Page 4/20...
2026-03-29 14:30:07,006 - INFO - Found 30 listings (Total unique: 120)
2026-03-29 14:30:09,579 - INFO - Processing Page 5/20...
2026-03-29 14:30:10,631 - INFO - Found 30 listings (Total unique: 150)
2026-03-29 14:30:13,209 - INFO - Processing Page 6/20...
2026-03-29 14:30:14,257 - INFO - Found 30 listings (Total unique:

In [ ]:
import csv
import time
import logging
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/fazwaz')
INPUT_CSV = BASE_DIR / 'fazwaz_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'fazwaz_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    
    return uc.Chrome(options=options, version_main=145)

def extract_content(driver: uc.Chrome) -> str:
    content_parts = []
    
    try:
        title = driver.find_element(By.CSS_SELECTOR, 'h1.project-name').text.strip()
        if title:
            content_parts.extend(["--- Listing Title ---", title])

        loc = driver.find_element(By.CSS_SELECTOR, '.project-location').text.strip()
        if loc:
            content_parts.append(f"Location: {loc}")
            
        prices = driver.find_elements(By.CSS_SELECTOR, '.gallery-project-price__item')
        if prices:
            content_parts.append("\n--- Price Information ---")
            for p in prices:
                content_parts.append(p.text.replace('\n', ' ').strip())
                
        info_elements = driver.find_elements(By.CSS_SELECTOR, '.property-info-element')
        if info_elements:
            content_parts.append("\n--- Property Info ---")
            info_texts = [e.text.replace('\n', ' ').strip() for e in info_elements if e.text.strip()]
            content_parts.append(" | ".join(info_texts))
            
        try:
            header_data = driver.find_element(By.CSS_SELECTOR, '.header-data-topic').text.strip()
            sibling_p = driver.find_element(By.XPATH, "//div[@class='header-data']/following-sibling::p[1]").text.strip()
            if header_data or sibling_p:
                content_parts.append("\n--- Header Data ---")
                if header_data: content_parts.append(header_data)
                if sibling_p: content_parts.append(sibling_p)
        except NoSuchElementException:
            pass

        try:
            more_link = driver.find_element(By.ID, 'moreLessLink')
            driver.execute_script("arguments[0].click();", more_link)
        except NoSuchElementException:
            pass

        try:
            about_title = driver.find_element(By.CSS_SELECTOR, '.about-project__title').text.strip()
            about_desc = driver.find_element(By.CSS_SELECTOR, '.about-project__text-description').text.strip()
            if about_title or about_desc:
                content_parts.append("\n--- About Project ---")
                if about_title: content_parts.append(about_title)
                if about_desc: content_parts.append(about_desc)
        except NoSuchElementException:
            pass

    except NoSuchElementException:
        pass

    return "\n".join(content_parts)

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 10)

    try:
        driver.get(urls[0])
        logging.info("Waiting 40 seconds for manual login...")
        time.sleep(40)

        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    if driver.current_url != url:
                        driver.get(url)
                    
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.project-name')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()